# Ejercicio 04: Archivos + Logs para SRE

Este notebook está listo para abrirse en Colab.

1. Ejecuta la celda de creación del archivo de log.
2. Ejecuta las celdas con las funciones.
3. Verifica el reporte final.

In [ ]:
log_lines = [
    "2026-05-23 08:01:12 INFO api-gateway: Request received /health",
    "2026-05-23 08:01:13 INFO auth-service: Token validated",
    "2026-05-23 08:01:15 ERROR db-primary: Connection timeout after 30s",
    "2026-05-23 08:01:16 WARN cache-redis: Memory usage at 85%",
    "2026-05-23 08:01:18 ERROR payment-svc: Transaction failed - insufficient funds",
    "2026-05-23 08:01:20 INFO api-gateway: Request received /users",
    "2026-05-23 08:01:22 WARN db-primary: Slow query detected (2.3s)",
    "2026-05-23 08:01:25 ERROR auth-service: Invalid token - expired",
    "2026-05-23 08:01:27 INFO cache-redis: Cache hit ratio 94%",
    "2026-05-23 08:01:30 ERROR db-primary: Connection timeout after 30s",
    "2026-05-23 08:01:32 WARN payment-svc: Retry attempt 2/3",
    "2026-05-23 08:01:35 INFO api-gateway: Health check OK",
]

with open('service.log', 'w') as f:
    for line in log_lines:
        f.write(line + '\n')

print('Archivo service.log creado.')

In [ ]:
def leer_log(file_name):
    try:
        with open(file_name, 'r') as f:
            return [line.strip() for line in f.readlines()]
    except FileNotFoundError:
        print(f'Error: El archivo {file_name} no existe.')
        return []

In [ ]:
def parsear_linea(log_string):
    partes = log_string.split(' ', 4)
    if len(partes) != 5:
        raise ValueError(f'Formato inválido: {log_string}')

    fecha = partes[0]
    hora = partes[1]
    severity = partes[2]
    service_part = partes[3]
    message = partes[4]

    if not service_part.endswith(':'):
        raise ValueError(f'Formato inválido del servicio: {log_string}')

    service = service_part[:-1]
    timestamp = f'{fecha} {hora}'

    return {
        'timestamp': timestamp,
        'severity': severity,
        'service': service,
        'message': message,
    }

In [ ]:
def generar_reporte(lineas):
    contador = {'ERROR': 0, 'WARN': 0, 'INFO': 0}
    servicios_error = set()

    for linea in lineas:
        if not linea:
            continue
        try:
            registro = parsear_linea(linea)
            severity = registro['severity']
            if severity in contador:
                contador[severity] += 1
            if severity == 'ERROR':
                servicios_error.add(registro['service'])
        except ValueError as err:
            print('Línea inválida:', err)

    total = sum(contador.values())
    print('=== REPORTE DE LOG ===')
    print(f'Total líneas: {total}')
    print(f"ERROR: {contador['ERROR']} | WARN: {contador['WARN']} | INFO: {contador['INFO']}")
    if servicios_error:
        print('Servicios con ERROR:')
        for servicio in sorted(servicios_error):
            print('-', servicio)

In [ ]:
lineas = leer_log('service.log')
generar_reporte(lineas)

lineas_no_existe = leer_log('no_existe.log')
generar_reporte(lineas_no_existe)